> 在线讲义：[第 9 章 · 训练循环：五行主干 + 三件护具](https://wikieden.github.io/pytorch-inside-out/lessons/09-training-loop.html) · 本书《PyTorch 从里到外》

# 第 9 章 · 训练循环：五行主干 + 三件护具

本章回答：一个训练步到底要写哪几行、顺序为什么不能乱？混合精度（autocast）、梯度裁剪、学习率调度分别加在哪、解决什么问题？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/09-training-loop.md`（讲解、动画、练习都在讲义里）。对应原文：00-overview.md §3；官方教程「Optimizing Model Parameters」（〔在线〕）；CS336 a1 handout §4–§5（优化器、学习率调度、梯度裁剪、训练循环）；本章数字在本机 torch 2.15.0 上跑过（代码见本章末尾"代码对照"）

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 拟合 y = 3x + 1 | 五行主干 | `fit_line()` | 步 0 loss 3.42、步 20 w = 3.000、b = 1.000 |
| 第三步 裁剪 | [3, 4] 缩到长度 1 | `clip_demo()` | 返回 5，梯度 [0.6, 0.8] |
| 第四步 warmup + cosine | 第 0/50/100/550/1000 步 | `scheduler_lrs()` | 0、1.5e-4、3e-4、1.5e-4、0 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `clip_demo`**（来自 `ch09_training_loop.py`）：下面要用到的小工具。

In [ ]:
def clip_demo(grad=(3.0, 4.0), max_norm: float = 1.0):
    p = nn.Parameter(torch.zeros(len(grad)))
    p.grad = torch.tensor(grad)
    n = torch.nn.utils.clip_grad_norm_([p], max_norm)
    return n.item(), p.grad.tolist()

**函数 `fit_line`**（来自 `ch09_training_loop.py`）：用 nn.Linear(1, 1) + SGD 拟合 y = 3x + 1；返回 {步: (更新前 loss, 更新后 w, b)}。

In [ ]:
def fit_line(steps: int = 21, lr: float = 0.5, seed: int = 0):
    """用 nn.Linear(1, 1) + SGD 拟合 y = 3x + 1；返回 {步: (更新前 loss, 更新后 w, b)}。"""
    torch.manual_seed(seed)
    X = torch.linspace(-1, 1, 32).unsqueeze(1)
    Y = 3 * X + 1
    m = nn.Linear(1, 1)
    opt = torch.optim.SGD(m.parameters(), lr=lr)
    log = {}
    for step in range(steps):
        loss = F.mse_loss(m(X), Y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        log[step] = (loss.item(), m.weight.item(), m.bias.item())
    return log

**函数 `warmup_cosine`**（来自 `ch09_training_loop.py`）：下面要用到的小工具。

In [ ]:
def warmup_cosine(step: int, base: float = 3e-4, warmup: int = 100, total: int = 1000) -> float:
    if step < warmup:
        return base * step / warmup
    return base * 0.5 * (1 + math.cos(math.pi * (step - warmup) / (total - warmup)))

**函数 `scheduler_lrs`**（来自 `ch09_training_loop.py`）：用 LambdaLR 真跑一遍，取指定步的学习率。

In [ ]:
def scheduler_lrs(base: float = 3e-4, warmup: int = 100, total: int = 1000, at=(0, 50, 100, 550, 1000)):
    """用 LambdaLR 真跑一遍，取指定步的学习率。"""
    opt = torch.optim.AdamW(nn.Linear(2, 2).parameters(), lr=base)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: warmup_cosine(s, base, warmup, total) / base)
    lrs = []
    for _ in range(total + 1):
        lrs.append(opt.param_groups[0]["lr"])
        opt.step()
        sch.step()
    return [lrs[s] for s in at]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
log = fit_line()

In [ ]:
print("\n第一步 拟合 y = 3x + 1：")
for s in (0, 1, 2, 5, 10, 20):
    loss, w, b = log[s]
    print(f"  步 {s:2d}: loss {loss:.4f}  w {w:.3f}  b {b:.3f}")

In [ ]:
print("\n第三步 裁剪 [3, 4] 到 1：", clip_demo())

In [ ]:
print("\n第四步 warmup + cosine 第 0/50/100/550/1000 步：", [f"{v:.2e}" for v in scheduler_lrs()])

## 3. 代码里才看得到的

1. **`LambdaLR` 的函数返回倍数，不是学习率本身**：所以要除以 base。
2. **裁剪后是 0.5999999 而不是 0.6**：浮点误差，测试用容差比较。

## 4. 自己改一改

- 把 `fit_line(lr=1.2)`，看 loss 是不是开始震荡。
- 把 `clip_demo` 的上限改成 10，看梯度还变不变。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 前向、loss、清梯度、反向、更新；清梯度只要在反向前；
> autocast 包前向，矩阵走 bf16；裁剪放在反向和更新之间，方向不变只缩长；
> 学习率先升后余弦降，调度跟在更新后。